[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/04_calculus_and_gradients.ipynb)

# Bài 04 — Giải tích cho ML: đạo hàm, gradient, chain rule

> Khoá AIVN có kế hoạch bài *"Giải tích cơ bản"* ở Tuần 1 nhưng chưa có nội dung — bài này bổ sung phần đó, tập trung vào những gì ML thực sự cần.

**🎯 Mục tiêu**
- Hiểu đạo hàm là "độ nhạy" của đầu ra theo đầu vào.
- Tính đạo hàm riêng, **gradient** và hiểu gradient chỉ hướng tăng nhanh nhất.
- **Chain rule** — nền tảng của backpropagation.
- Jacobian, Hessian, tính lồi (convexity).
- Kiểm tra gradient bằng **sai phân số** (gradient check).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.utils import numerical_gradient, relative_error
from ml_course.visualization import set_style

set_style()

## 1. Đạo hàm — độ nhạy

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

$f'(x)$ cho biết: nếu tăng $x$ một lượng rất nhỏ $\Delta x$, thì $f$ thay đổi xấp xỉ $f'(x)\,\Delta x$. Về hình học, đó là **độ dốc của tiếp tuyến**.

| Hàm | Đạo hàm | Gặp ở đâu |
|---|---|---|
| $x^n$ | $n x^{n-1}$ | MSE: $(y - \hat y)^2$ |
| $e^x$ | $e^x$ | softmax, sigmoid |
| $\ln x$ | $1/x$ | log-likelihood, cross-entropy |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | $\sigma(x)(1-\sigma(x))$ | logistic regression, neural net |
| $\tanh x$ | $1 - \tanh^2 x$ | RNN |
| $\max(0, x)$ (ReLU) | $\mathbb 1[x > 0]$ | deep learning |
| $\lvert x\rvert$ | $\operatorname{sign}(x)$ (không khả vi tại 0) | L1 loss, Lasso |

Quy tắc: $(f+g)' = f' + g'$, $(cf)' = cf'$, $(fg)' = f'g + fg'$, $(f/g)' = (f'g - fg')/g^2$.

In [ ]:
f = lambda x: x**3 - 3 * x
df = lambda x: 3 * x**2 - 3

xs = np.linspace(-2.5, 2.5, 300)
fig, ax = plt.subplots()
ax.plot(xs, f(xs), label="f(x) = x³ − 3x")
for x0 in [-1.8, 0.0, 1.0]:
    tangent = f(x0) + df(x0) * (xs - x0)
    ax.plot(xs, tangent, "--", lw=1)
    ax.plot(x0, f(x0), "o")
    ax.annotate(f"f'={df(x0):.1f}", (x0, f(x0)), textcoords="offset points", xytext=(5, 8))
ax.set_ylim(-5, 5)
ax.legend()
ax.set_title("Đạo hàm = độ dốc tiếp tuyến (f' = 0 tại cực trị)")
plt.show()

**Đạo hàm số (numerical derivative)** — xấp xỉ bằng sai phân trung tâm, chính xác hơn sai phân một phía:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

In [ ]:
x0 = 1.3
for h in [1e-1, 1e-3, 1e-5, 1e-8, 1e-12]:
    forward = (f(x0 + h) - f(x0)) / h
    central = (f(x0 + h) - f(x0 - h)) / (2 * h)
    print(f"h={h:<6g} sai số một phía={abs(forward - df(x0)):.2e}   sai số trung tâm={abs(central - df(x0)):.2e}")

Chú ý: $h$ quá nhỏ ($10^{-12}$) lại làm sai số **tăng** do giới hạn độ chính xác của số thực. Thường chọn $h \approx 10^{-5}$.

## 2. Đạo hàm riêng và gradient

Với hàm nhiều biến $f(\mathbf x) = f(x_1, \dots, x_d)$, **đạo hàm riêng** $\frac{\partial f}{\partial x_j}$ là đạo hàm theo $x_j$ khi giữ các biến khác cố định. **Gradient** gom tất cả đạo hàm riêng thành một vector:

$$\nabla f(\mathbf x) = \left[\frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_d}\right]^\top$$

**Tính chất then chốt:** $\nabla f$ chỉ **hướng tăng nhanh nhất** của $f$; $-\nabla f$ là hướng **giảm nhanh nhất** → ý tưởng của gradient descent (bài 05). Gradient vuông góc với đường đồng mức.

Ví dụ: $f(x, y) = x^2 + 3y^2$ ⇒ $\nabla f = [2x, 6y]$.

In [ ]:
f2 = lambda x, y: x**2 + 3 * y**2
grad_f2 = lambda x, y: np.array([2 * x, 6 * y])

X, Y = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-2, 2, 200))
fig, ax = plt.subplots(figsize=(7, 4.8))
cs = ax.contour(X, Y, f2(X, Y), levels=15, cmap="viridis")
ax.clabel(cs, fontsize=7)
gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 9), np.linspace(-1.6, 1.6, 7))
G = grad_f2(gx, gy)
ax.quiver(gx, gy, -G[0], -G[1], color="red", alpha=0.7, label="−∇f (hướng giảm nhanh nhất)")
ax.set_aspect("equal")
ax.legend(loc="upper right")
ax.set_title("−∇f vuông góc với đường đồng mức, chỉ về điểm cực tiểu")
plt.show()

## 3. Chain rule — trái tim của backpropagation

Nếu $y = f(u)$ và $u = g(x)$ thì:

$$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}$$

Ví dụ thực tế: loss của **logistic regression** cho một mẫu, với $z = wx + b$, $p = \sigma(z)$, $L = -[y\log p + (1-y)\log(1-p)]$:

$$\frac{\partial L}{\partial w} = \underbrace{\frac{\partial L}{\partial p}}_{\frac{p - y}{p(1-p)}}\cdot\underbrace{\frac{\partial p}{\partial z}}_{p(1-p)}\cdot\underbrace{\frac{\partial z}{\partial w}}_{x} = (p - y)\,x$$

Kết quả gọn gàng $(p - y)x$ — "sai số × đầu vào" — sẽ xuất hiện lại ở rất nhiều model.

In [ ]:
def logistic_loss(w, b, x, y):
    p = 1 / (1 + np.exp(-(w * x + b)))
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


w, b, x, y = 0.7, -0.2, 2.5, 1
p = 1 / (1 + np.exp(-(w * x + b)))
analytic = (p - y) * x
numeric = numerical_gradient(lambda v: logistic_loss(v[0], b, x, y), np.array([w]))[0]
print(f"chain rule: {analytic:.8f} | sai phân số: {numeric:.8f}")

Với nhiều tầng hàm lồng nhau (neural network), backpropagation chỉ là áp dụng chain rule **từ đầu ra ngược về đầu vào**, tái sử dụng các tích đã tính — chương 27 sẽ tự xây một "mini autograd" để thấy điều này.

## 4. Gradient của các biểu thức ma trận thường gặp

Bảng "matrix calculus" tối thiểu cho ML (với $\mathbf w, \mathbf a \in \mathbb{R}^d$, $X \in \mathbb{R}^{n\times d}$, $A$ đối xứng):

| $f(\mathbf w)$ | $\nabla_{\mathbf w} f$ |
|---|---|
| $\mathbf a^\top\mathbf w$ | $\mathbf a$ |
| $\mathbf w^\top\mathbf w = \lVert\mathbf w\rVert^2$ | $2\mathbf w$ |
| $\mathbf w^\top A\mathbf w$ | $2A\mathbf w$ |
| $\lVert X\mathbf w - \mathbf y\rVert^2$ | $2X^\top(X\mathbf w - \mathbf y)$ |
| $\frac{1}{n}\lVert X\mathbf w - \mathbf y\rVert^2$ (MSE) | $\frac{2}{n}X^\top(X\mathbf w - \mathbf y)$ |

Dòng cuối là gradient của Linear Regression. Đặt bằng 0 ⇒ $X^\top X\mathbf w = X^\top\mathbf y$ — **normal equation** (chương 06). Kiểm chứng bằng sai phân số:

In [ ]:
rng = np.random.default_rng(0)
Xm, ym, wm = rng.normal(size=(20, 3)), rng.normal(size=20), rng.normal(size=3)
mse = lambda w: np.mean((Xm @ w - ym) ** 2)
analytic = 2 / len(ym) * Xm.T @ (Xm @ wm - ym)
numeric = numerical_gradient(mse, wm.copy())
print("analytic:", analytic.round(6))
print("numeric :", numeric.round(6))
print("relative error:", f"{relative_error(analytic, numeric):.2e}", "→ < 1e-7 là gradient đúng")

> **Gradient check** là kỹ thuật debug quan trọng: mỗi khi tự viết gradient (chương 06, 07, 27), hãy so sánh với sai phân số trên dữ liệu nhỏ. Relative error > $10^{-3}$ gần như chắc chắn có bug.

## 5. Jacobian và Hessian

- **Jacobian** $J$: khi hàm trả về **vector** $\mathbf f: \mathbb R^d \to \mathbb R^m$, $J_{ij} = \frac{\partial f_i}{\partial x_j}$ (ma trận $m \times d$). Ví dụ: đạo hàm của softmax, của một layer neural network.
- **Hessian** $H$: ma trận đạo hàm **bậc hai** của hàm vô hướng, $H_{ij} = \frac{\partial^2 f}{\partial x_i\partial x_j}$ — mô tả **độ cong**.

Với $f(x, y) = x^2 + 3y^2$: $H = \begin{bmatrix}2 & 0\\0 & 6\end{bmatrix}$ — cong gấp 3 lần theo trục $y$ → đường đồng mức là elip dẹt → gradient descent sẽ "zig-zag" (bài 05). Tỉ số trị riêng lớn nhất/nhỏ nhất của Hessian chính là **condition number** của bài toán tối ưu.

In [ ]:
def numerical_hessian(f, x, eps=1e-4):
    d = len(x)
    H = np.zeros((d, d))
    for i in range(d):
        e = np.zeros(d)
        e[i] = eps
        H[:, i] = (numerical_gradient(f, x + e) - numerical_gradient(f, x - e)) / (2 * eps)
    return H


H = numerical_hessian(lambda v: v[0] ** 2 + 3 * v[1] ** 2, np.array([1.0, 1.0]))
print(H.round(4))
print("trị riêng:", np.linalg.eigvalsh(H).round(4), "→ đều dương ⇒ hàm lồi")

## 6. Hàm lồi (convex) — vì sao quan trọng?

Hàm $f$ **lồi** nếu đoạn thẳng nối hai điểm bất kỳ trên đồ thị luôn nằm **trên** đồ thị. Tương đương (hàm khả vi 2 lần): Hessian **nửa xác định dương** (mọi trị riêng ≥ 0).

**Hệ quả quan trọng:** với hàm lồi, **mọi cực tiểu địa phương đều là cực tiểu toàn cục** → gradient descent tìm được nghiệm tốt nhất.

| Lồi ✅ | Không lồi ❌ |
|---|---|
| MSE của Linear Regression | loss của Neural Network |
| Cross-entropy của Logistic Regression | K-Means (theo vị trí tâm) |
| Hinge loss của SVM, Ridge, Lasso | Matrix factorization |

In [ ]:
xs = np.linspace(-2.2, 2.2, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(xs, xs**2)
axes[0].plot([-1.5, 2], [2.25, 4], "r--o")
axes[0].set_title("Lồi: x² — một cực tiểu duy nhất")
g = xs**4 - 3 * xs**2 + 0.6 * xs
axes[1].plot(xs, g)
axes[1].set_title("Không lồi: nhiều cực tiểu địa phương")
plt.show()

## 🧠 Kiểm tra nhanh

1. $f(w) = (3w - 2)^2$. Tính $f'(w)$ bằng chain rule. $f'$ bằng 0 tại đâu?
2. Gradient của $f(x, y) = x^2y + y^3$ tại $(1, 2)$?
3. Vì sao gradient descent đi theo $-\nabla f$ chứ không phải $+\nabla f$?
4. Hessian có một trị riêng âm nói gì về hình dạng hàm tại điểm đó?

## 📌 Tóm tắt

- Đạo hàm = độ nhạy; gradient = vector các đạo hàm riêng, chỉ hướng tăng nhanh nhất.
- Chain rule cho phép tính gradient của hàm lồng nhau → backpropagation.
- $\nabla_{\mathbf w}\frac1n\lVert X\mathbf w-\mathbf y\rVert^2 = \frac2n X^\top(X\mathbf w-\mathbf y)$ — gradient quan trọng nhất của khoá học.
- Luôn **gradient check** bằng sai phân trung tâm khi tự viết gradient.
- Hàm lồi: cực tiểu địa phương = toàn cục; Hessian mô tả độ cong và độ khó của bài toán tối ưu.

## 📚 Tài liệu tham khảo

- 3Blue1Brown — *Essence of Calculus*
- Deisenroth, Faisal, Ong — *Mathematics for Machine Learning*, chương 5 (miễn phí: https://mml-book.github.io)
- The Matrix Cookbook — https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf